# HCLA 통합 실험 — Multi-model + Generalization + SHAP + Qwen + Llama

기존 XGBoost 단일 실험을 다음 구조로 확장한 버전입니다.

- **RQ1 detector 비교:** Logistic Regression, Random Forest, XGBoost, LightGBM, Tabular ResNet, FT-Transformer
- **E1:** Random + Address Included
- **E2:** Random + Address Removed
- **E3:** Temporal(2020–2022 → 2023–2024) + Address Removed
- **E4:** Temporal + Address-disjoint + Address Removed
- **3 seeds:** 42, 52, 62
- **SHAP:** global beeswarm, global bar, local waterfall, Top-k evidence
- **LLM 생성:** Qwen
- **LLM 검증/Judge:** Llama 3.3 70B via OpenRouter
- **RQ2:** Evidence groundedness
- **RQ3:** Explanation format comparison
- **RQ4:** Component ablation
- **RQ5:** Stability / ambiguous / adversarial robustness

논문 최종 실행 전에는 `FAST_MODE=False`, `RQ2_PILOT=False`, `PILOT_MODE=False`로 설정하세요.

## 0. 설치

In [ ]:
!pip -q install -U numpy pandas scikit-learn matplotlib xgboost lightgbm shap torch openai google-genai

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Imports / 설정

In [ ]:
import os, json, re, time, random, pickle, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, average_precision_score, roc_auc_score,
    confusion_matrix, precision_recall_curve
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import shap

warnings.filterwarnings("ignore")

SEEDS = [42, 52, 62]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_CUDA = torch.cuda.is_available()
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    GPU_NAME = torch.cuda.get_device_name(0)
else:
    GPU_NAME = "CPU only"

MODEL_NAMES = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "TabularResNet",
    "FTTransformer",
]

FAST_MODE = False
DEEP_EPOCHS = 5 if FAST_MODE else 20
DEEP_BATCH_SIZE = 4096
DEEP_PATIENCE = 3 if FAST_MODE else 5

# AUTO: E4 Macro-F1 -> PR-AUC 순으로 detector 자동 선택
# 필요하면 "XGBoost"처럼 직접 지정
HCLA_MODEL = "AUTO"

print("DEVICE:", DEVICE)
print("GPU:", GPU_NAME)
if not USE_CUDA:
    print("WARNING: GPU가 감지되지 않았습니다. Colab > 런타임 > 런타임 유형 변경 > GPU를 선택하세요.")
print("MODELS:", MODEL_NAMES)

### GPU 실행 설정
- **XGBoost**: CUDA가 있으면 `device="cuda"`로 GPU 학습
- **TabularResNet / FTTransformer**: PyTorch CUDA 사용
- **Logistic Regression / Random Forest / LightGBM**: CPU 유지 (Colab 기본 LightGBM 빌드의 GPU 호환성 문제를 피하기 위함)
- Colab에서 먼저 **런타임 → 런타임 유형 변경 → GPU**를 선택하세요.


## 2. 데이터

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Data/BTC/ETH MixingTornadoCash DF.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Results/HCLA_MultiModel_Qwen_Llama_OpenRouter_SHAP"

In [ ]:

# ============================================================
# CHECKPOINT / RESUME UTILITIES
# ============================================================
import os, json, pickle, tempfile
import pandas as pd

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "_checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def atomic_pickle_dump(obj, path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", dir=os.path.dirname(path))
    os.close(fd)
    try:
        with open(tmp, "wb") as f:
            pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

def atomic_json_dump(obj, path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", dir=os.path.dirname(path))
    os.close(fd)
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

def atomic_csv_save(df, path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", suffix=".csv", dir=os.path.dirname(path))
    os.close(fd)
    try:
        df.to_csv(tmp, index=False, encoding="utf-8-sig")
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

def load_csv_or_empty(path, columns=None):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ checkpoint loaded: {os.path.basename(path)} ({len(df):,} rows)")
        return df
    return pd.DataFrame(columns=columns)

def keyset(df, cols):
    if df is None or len(df) == 0 or any(c not in df.columns for c in cols):
        return set()
    return set(map(tuple, df[cols].astype(str).itertuples(index=False, name=None)))

print("Checkpoint directory:", CHECKPOINT_DIR)


In [ ]:



os.makedirs(OUTPUT_DIR, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
assert "label" in df_raw.columns
assert "Date" in df_raw.columns

print("shape:", df_raw.shape)
print(df_raw["label"].value_counts(dropna=False).sort_index())

## 3. 전처리

In [ ]:
DATE_COL = "Date"
LABEL_COL = "label"
ADDRESS_COLS = ["Receiving Address", "Counterparty Address"]

ALWAYS_DROP_COLS = [
    "Hash",
    "Source",
    "Counterparty Cluster Name",
    "Counterparty Shared Name",
    "Counterparty Category",
]

df = df_raw.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

def add_time_features(frame):
    out = frame.copy()
    dt = pd.to_datetime(out[DATE_COL], errors="coerce")
    out["Year"] = dt.dt.year
    out["Month"] = dt.dt.month
    out["Day"] = dt.dt.day
    out["Hour"] = dt.dt.hour
    out["DayOfWeek"] = dt.dt.dayofweek
    return out

def build_feature_frames(train_df, test_df, include_address):
    tr = add_time_features(train_df)
    te = add_time_features(test_df)

    y_train = tr[LABEL_COL].astype(int).copy()
    y_test = te[LABEL_COL].astype(int).copy()

    drop_cols = [LABEL_COL, DATE_COL] + ALWAYS_DROP_COLS
    if not include_address:
        drop_cols += ADDRESS_COLS

    X_train = tr.drop(columns=drop_cols, errors="ignore").copy()
    X_test = te.drop(columns=drop_cols, errors="ignore").copy()

    if include_address:
        addr_cols = [c for c in ADDRESS_COLS if c in X_train.columns]
        if addr_cols:
            X_train[addr_cols] = X_train[addr_cols].fillna("__MISSING__").astype(str)
            X_test[addr_cols] = X_test[addr_cols].fillna("__MISSING__").astype(str)

            enc = OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
            X_train[addr_cols] = enc.fit_transform(X_train[addr_cols])
            X_test[addr_cols] = enc.transform(X_test[addr_cols])

    # 남는 텍스트 컬럼은 label proxy 방지를 위해 제거
    obj_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    if obj_cols:
        X_train = X_train.drop(columns=obj_cols, errors="ignore")
        X_test = X_test.drop(columns=obj_cols, errors="ignore")

    X_test = X_test.reindex(columns=X_train.columns)
    return X_train, X_test, y_train, y_test

print("usable rows:", len(df))
display(df.groupby(LABEL_COL)[DATE_COL].agg(["min", "max"]))

## 4. Split

In [ ]:
def split_random(data, split_seed):
    return tuple(
        x.copy() for x in train_test_split(
            data,
            test_size=0.30,
            random_state=split_seed,
            stratify=data[LABEL_COL],
        )
    )

def split_temporal(data):
    train_df = data[
        (data[DATE_COL] >= "2020-01-01") &
        (data[DATE_COL] <= "2022-12-31 23:59:59")
    ].copy()
    test_df = data[
        (data[DATE_COL] >= "2023-01-01") &
        (data[DATE_COL] <= "2024-12-31 23:59:59")
    ].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        raise ValueError("Temporal split이 비었습니다.")
    return train_df, test_df

def split_temporal_address_disjoint(data):
    train_df, test_candidate = split_temporal(data)

    train_addresses = set()
    for c in ADDRESS_COLS:
        train_addresses.update(train_df[c].dropna().astype(str).unique().tolist())

    recv_unseen = ~test_candidate["Receiving Address"].fillna("__MISSING__").astype(str).isin(train_addresses)
    cp_unseen = ~test_candidate["Counterparty Address"].fillna("__MISSING__").astype(str).isin(train_addresses)
    test_df = test_candidate[recv_unseen & cp_unseen].copy()

    test_addresses = set()
    for c in ADDRESS_COLS:
        test_addresses.update(test_df[c].dropna().astype(str).unique().tolist())

    overlap = train_addresses.intersection(test_addresses)
    assert len(overlap) == 0
    if len(test_df) == 0:
        raise ValueError("Address-disjoint test가 0건입니다.")

    print("Temporal candidate:", len(test_candidate))
    print("Address-disjoint test:", len(test_df))
    print("Address overlap:", len(overlap))
    return train_df, test_df

## 5. 평가 지표

In [ ]:
def evaluate_binary(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "n_test": len(y_true),
        "normal_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "normal_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "normal_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "anomaly_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "anomaly_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "anomaly_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

## 6. Tabular ResNet / FT-Transformer

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, d),
            nn.BatchNorm1d(d),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d, d),
            nn.BatchNorm1d(d),
        )
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(x + self.net(x))

class TabularResNetNet(nn.Module):
    def __init__(self, n_features, d=128, n_blocks=3, dropout=0.1):
        super().__init__()
        self.input = nn.Sequential(
            nn.Linear(n_features, d),
            nn.BatchNorm1d(d),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(*[
            ResidualBlock(d, dropout) for _ in range(n_blocks)
        ])
        self.head = nn.Linear(d, 1)

    def forward(self, x):
        return self.head(self.blocks(self.input(x))).squeeze(1)

class NumericTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))

    def forward(self, x):
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

class FTTransformerNet(nn.Module):
    def __init__(self, n_features, d_token=64, n_heads=8, n_layers=3, dropout=0.1):
        super().__init__()
        self.tokenizer = NumericTokenizer(n_features, d_token)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Linear(d_token, 1)

    def forward(self, x):
        tok = self.tokenizer(x)
        cls = self.cls.expand(x.shape[0], -1, -1)
        z = self.encoder(torch.cat([cls, tok], dim=1))
        return self.head(self.norm(z[:, 0])).squeeze(1)

class TorchBinaryClassifier:
    def __init__(self, architecture, seed=42, epochs=20, batch_size=4096, patience=5):
        self.architecture = architecture
        self.seed = seed
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.scaler = StandardScaler()
        self.model = None

    def _set_seed(self):
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def fit(self, X, y):
        self._set_seed()
        Xs = self.scaler.fit_transform(X).astype(np.float32)
        y = np.asarray(y).astype(np.float32)

        idx = np.arange(len(y))
        tr_idx, va_idx = train_test_split(
            idx, test_size=0.10, random_state=self.seed, stratify=y.astype(int)
        )

        train_loader = DataLoader(
            TensorDataset(
                torch.tensor(Xs[tr_idx], dtype=torch.float32),
                torch.tensor(y[tr_idx], dtype=torch.float32),
            ),
            batch_size=self.batch_size,
            shuffle=True,
        )
        val_loader = DataLoader(
            TensorDataset(
                torch.tensor(Xs[va_idx], dtype=torch.float32),
                torch.tensor(y[va_idx], dtype=torch.float32),
            ),
            batch_size=self.batch_size,
            shuffle=False,
        )

        n_features = Xs.shape[1]
        if self.architecture == "TabularResNet":
            self.model = TabularResNetNet(n_features).to(DEVICE)
        else:
            self.model = FTTransformerNet(n_features).to(DEVICE)

        n_pos = max(float((y[tr_idx] == 1).sum()), 1.0)
        n_neg = float((y[tr_idx] == 0).sum())
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([n_neg / n_pos], device=DEVICE)
        )
        optimizer = torch.optim.AdamW(
            self.model.parameters(), lr=1e-3, weight_decay=1e-5
        )

        best_loss = float("inf")
        best_state = None
        bad_epochs = 0

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(self.model(xb), yb)
                loss.backward()
                optimizer.step()

            self.model.eval()
            val_losses = []
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    val_losses.append(criterion(self.model(xb), yb).item())

            val_loss = float(np.mean(val_losses))
            if val_loss < best_loss - 1e-5:
                best_loss = val_loss
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in self.model.state_dict().items()
                }
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= self.patience:
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state)
        return self

    def predict_proba(self, X):
        Xs = self.scaler.transform(X).astype(np.float32)
        loader = DataLoader(
            TensorDataset(torch.tensor(Xs, dtype=torch.float32)),
            batch_size=self.batch_size,
            shuffle=False,
        )
        self.model.eval()
        probs = []
        with torch.no_grad():
            for (xb,) in loader:
                p = torch.sigmoid(self.model(xb.to(DEVICE))).cpu().numpy()
                probs.append(p)
        p1 = np.concatenate(probs)
        return np.column_stack([1 - p1, p1])

## 7. 공통 Model Factory / Runner

In [ ]:
XGB_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=-1,
    eval_metric="logloss",
    tree_method="hist",
    device="cuda" if USE_CUDA else "cpu",
)

LGBM_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=-1,
    verbosity=-1,
)

def make_model(model_name, seed, y_train):
    y_train = np.asarray(y_train)
    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())
    spw = n_neg / max(n_pos, 1)

    if model_name == "LogisticRegression":
        return LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=seed,
        )
    if model_name == "RandomForest":
        return RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=seed,
        )
    if model_name == "XGBoost":
        return XGBClassifier(
            **XGB_PARAMS,
            random_state=seed,
            scale_pos_weight=spw,
        )
    if model_name == "LightGBM":
        return LGBMClassifier(
            **LGBM_PARAMS,
            random_state=seed,
            scale_pos_weight=spw,
        )
    if model_name in ["TabularResNet", "FTTransformer"]:
        return TorchBinaryClassifier(
            architecture=model_name,
            seed=seed,
            epochs=DEEP_EPOCHS,
            batch_size=DEEP_BATCH_SIZE,
            patience=DEEP_PATIENCE,
        )
    raise ValueError(model_name)

def run_model_experiment(
    model_name,
    experiment_name,
    train_df,
    test_df,
    include_address,
    model_seed,
    split_seed=None,
):
    X_train, X_test, y_train, y_test = build_feature_frames(
        train_df, test_df, include_address=include_address
    )

    imputer = SimpleImputer(strategy="median")
    X_train_arr = imputer.fit_transform(X_train)
    X_test_arr = imputer.transform(X_test)

    scaler = None
    fit_train, fit_test = X_train_arr, X_test_arr

    if model_name == "LogisticRegression":
        scaler = StandardScaler()
        fit_train = scaler.fit_transform(X_train_arr)
        fit_test = scaler.transform(X_test_arr)

    model = make_model(model_name, model_seed, y_train)
    model.fit(fit_train, y_train.to_numpy())

    y_prob = model.predict_proba(fit_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = evaluate_binary(y_test.to_numpy(), y_pred, y_prob)
    metrics.update({
        "model": model_name,
        "experiment": experiment_name,
        "seed": model_seed,
        "split_seed": split_seed,
        "include_address": include_address,
        "n_train": len(y_train),
        "n_features": X_train.shape[1],
        "train_normal": int((y_train == 0).sum()),
        "train_anomaly": int((y_train == 1).sum()),
        "test_normal": int((y_test == 0).sum()),
        "test_anomaly": int((y_test == 1).sum()),
    })

    return {
        "model_name": model_name,
        "name": experiment_name,
        "model": model,
        "imputer": imputer,
        "scaler": scaler,
        "feature_names": X_train.columns.tolist(),
        "y_test": y_test.reset_index(drop=True),
        "y_pred": np.asarray(y_pred),
        "y_prob": np.asarray(y_prob),
        "metrics": metrics,
    }

def model_input_from_raw(run, X_df):
    arr = run["imputer"].transform(X_df)
    if run["scaler"] is not None:
        arr = run["scaler"].transform(arr)
    return arr

def get_result_run(results, model_name, experiment_name, seed):
    x = [
        r for r in results
        if r["model_name"] == model_name
        and r["name"] == experiment_name
        and r["metrics"]["seed"] == seed
    ]
    if len(x) != 1:
        raise ValueError(f"Expected one run, found {len(x)}")
    return x[0]

## 8. RQ1 — 6 Models × 4 Conditions × 3 Seeds

In [ ]:

import time

MODEL_NAMES = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
]

RQ1_RUN_CKPT = os.path.join(CHECKPOINT_DIR, "rq1_runs.pkl")

if os.path.exists(RQ1_RUN_CKPT):
    with open(RQ1_RUN_CKPT, "rb") as f:
        results = pickle.load(f)
    print(f"✅ RQ1 resume: {len(results)} runs loaded")
else:
    results = []

def rq1_key(r):
    seed = r.get("metrics", {}).get("seed", r.get("model_seed", ""))
    return (str(r.get("model_name")), str(r.get("name")), str(seed))

done = {rq1_key(r) for r in results}

def rq1_run_one(model_name, exp_name, train_df, test_df,
                include_address, model_seed, split_seed=None):
    key = (str(model_name), str(exp_name), str(model_seed))

    if key in done:
        print("⏭️ RQ1 SKIP:", key)
        return

    print("▶ RQ1:", key)
    t0 = time.perf_counter()

    kwargs = dict(
        model_name=model_name,
        experiment_name=exp_name,
        train_df=train_df,
        test_df=test_df,
        include_address=include_address,
        model_seed=model_seed,
    )
    if split_seed is not None:
        kwargs["split_seed"] = split_seed

    r = run_model_experiment(**kwargs)
    r["runtime_sec"] = time.perf_counter() - t0

    results.append(r)
    done.add(key)
    atomic_pickle_dump(results, RQ1_RUN_CKPT)

    print("✅ RQ1 saved:", key)

for seed in SEEDS:
    random_train, random_test = split_random(df, seed)
    for model_name in MODEL_NAMES:
        rq1_run_one(
            model_name, "E1_Random_AddressIncluded",
            random_train, random_test, True, seed, seed
        )
        rq1_run_one(
            model_name, "E2_Random_AddressRemoved",
            random_train, random_test, False, seed, seed
        )

temporal_train, temporal_test = split_temporal(df)
for seed in SEEDS:
    for model_name in MODEL_NAMES:
        rq1_run_one(
            model_name, "E3_Temporal_AddressRemoved",
            temporal_train, temporal_test, False, seed
        )

disjoint_train, disjoint_test = split_temporal_address_disjoint(df)
for seed in SEEDS:
    for model_name in MODEL_NAMES:
        rq1_run_one(
            model_name, "E4_Temporal_AddressDisjoint_AddressRemoved",
            disjoint_train, disjoint_test, False, seed
        )

print("RQ1 completed:", len(results), "/", len(SEEDS) * len(MODEL_NAMES) * 4)


## 9. RQ1 결과표 / Generalization Drop

In [ ]:
results_df = pd.DataFrame([r["metrics"] for r in results])

metric_cols = [
    "accuracy",
    "normal_precision", "normal_recall", "normal_f1",
    "anomaly_precision", "anomaly_recall", "anomaly_f1",
    "macro_f1", "balanced_accuracy", "pr_auc", "roc_auc",
    "fpr", "fnr",
]

summary_mean = (
    results_df.groupby(["model", "experiment"])[metric_cols]
    .mean()
    .reset_index()
)
summary_sd = (
    results_df.groupby(["model", "experiment"])[metric_cols]
    .std(ddof=1)
    .reset_index()
)

display(
    summary_mean[
        [
            "model", "experiment",
            "macro_f1", "balanced_accuracy", "pr_auc",
            "anomaly_precision", "anomaly_recall", "anomaly_f1"
        ]
    ]
    .sort_values(["experiment", "macro_f1"], ascending=[True, False])
    .round(4)
)

paper_rows = []
for _, m in summary_mean.iterrows():
    s = summary_sd[
        (summary_sd["model"] == m["model"]) &
        (summary_sd["experiment"] == m["experiment"])
    ].iloc[0]

    row = {"model": m["model"], "experiment": m["experiment"]}
    for metric in metric_cols:
        row[metric] = f"{m[metric]:.4f} ± {s[metric]:.4f}"
    paper_rows.append(row)

paper_summary = pd.DataFrame(paper_rows)
display(paper_summary)

pivot = summary_mean.pivot(
    index="model",
    columns="experiment",
    values=["macro_f1", "pr_auc", "anomaly_f1"],
)

drop_rows = []
for model_name in pivot.index:
    e1 = "E1_Random_AddressIncluded"
    e4 = "E4_Temporal_AddressDisjoint_AddressRemoved"
    drop_rows.append({
        "model": model_name,
        "macro_f1_E1": pivot.loc[model_name, ("macro_f1", e1)],
        "macro_f1_E4": pivot.loc[model_name, ("macro_f1", e4)],
        "macro_f1_drop": (
            pivot.loc[model_name, ("macro_f1", e1)]
            - pivot.loc[model_name, ("macro_f1", e4)]
        ),
        "pr_auc_drop": (
            pivot.loc[model_name, ("pr_auc", e1)]
            - pivot.loc[model_name, ("pr_auc", e4)]
        ),
        "anomaly_f1_drop": (
            pivot.loc[model_name, ("anomaly_f1", e1)]
            - pivot.loc[model_name, ("anomaly_f1", e4)]
        ),
    })

generalization_drop_df = pd.DataFrame(drop_rows).sort_values("macro_f1_drop")
display(generalization_drop_df.round(4))

## 10. HCLA detector 선택

In [ ]:
e4_rank = (
    summary_mean[
        summary_mean["experiment"]
        == "E4_Temporal_AddressDisjoint_AddressRemoved"
    ]
    .sort_values(
        [
            "pr_auc",
            "anomaly_recall",
            "anomaly_f1",
            "macro_f1"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    e4_rank[
        [
            "model",
            "pr_auc",
            "anomaly_recall",
            "anomaly_f1",
            "macro_f1",
            "balanced_accuracy"
        ]
    ].round(4)
)

if HCLA_MODEL == "AUTO":
    SELECTED_MODEL = e4_rank.iloc[0]["model"]
else:
    SELECTED_MODEL = HCLA_MODEL

print("SELECTED_MODEL:", SELECTED_MODEL)

## 11. E4 Precision–Recall Curve

In [ ]:
plt.figure(figsize=(10, 7))

for model_name in MODEL_NAMES:
    r = get_result_run(
        results,
        model_name,
        "E4_Temporal_AddressDisjoint_AddressRemoved",
        42
    )
    precision, recall, _ = precision_recall_curve(r["y_test"], r["y_prob"])
    plt.plot(
        recall,
        precision,
        label=f"{model_name} (AP={r['metrics']['pr_auc']:.3f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("E4 Precision–Recall Curves")
plt.legend(fontsize=8)
plt.grid(alpha=0.2)
plt.tight_layout()

path = os.path.join(OUTPUT_DIR, "rq1_e4_pr_curves.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(path)

## 12. RQ1 저장

In [ ]:
RQ1_PKL_PATH = os.path.join(
    OUTPUT_DIR,
    "rq1_multimodel_results_3seeds.pkl"
)

results_df.to_csv(
    os.path.join(OUTPUT_DIR, "rq1_all_runs.csv"),
    index=False,
    encoding="utf-8-sig"
)
summary_mean.to_csv(
    os.path.join(OUTPUT_DIR, "rq1_mean.csv"),
    index=False,
    encoding="utf-8-sig"
)
summary_sd.to_csv(
    os.path.join(OUTPUT_DIR, "rq1_sd.csv"),
    index=False,
    encoding="utf-8-sig"
)
paper_summary.to_csv(
    os.path.join(OUTPUT_DIR, "rq1_paper_mean_sd.csv"),
    index=False,
    encoding="utf-8-sig"
)
generalization_drop_df.to_csv(
    os.path.join(OUTPUT_DIR, "rq1_generalization_drop.csv"),
    index=False,
    encoding="utf-8-sig"
)

with open(RQ1_PKL_PATH, "wb") as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

print(RQ1_PKL_PATH)
atomic_pickle_dump(results, RQ1_RUN_CKPT)
print("✅ RQ1 checkpoint synchronized")


임계값별 precision–recall curve (threshold별 PR curve 실험)

In [ ]:
import pickle
import os, json, re, time, random, pickle, warnings
from collections import Counter

RQ1_PKL_PATH = os.path.join(
    OUTPUT_DIR,
    "rq1_multimodel_results_3seeds.pkl"
)

In [ ]:

with open(RQ1_PKL_PATH, "rb") as f:
    rq1_results = pickle.load(f)

print(type(rq1_results))

if isinstance(rq1_results, dict):
    print(rq1_results.keys())

elif isinstance(rq1_results, list):
    print("길이:", len(rq1_results))
    print("첫 번째 결과 type:", type(rq1_results[0]))

    if isinstance(rq1_results[0], dict):
        print("저장된 key:", rq1_results[0].keys())
        print(rq1_results[0])

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# =========================================================
# 1. PKL Load
# =========================================================
RQ1_PKL_PATH = os.path.join(
    OUTPUT_DIR,
    "rq1_multimodel_results_3seeds.pkl"
)

with open(RQ1_PKL_PATH, "rb") as f:
    rq1_results = pickle.load(f)

print("Loaded runs:", len(rq1_results))


# =========================================================
# 2. 분석 대상
# =========================================================
TARGET_EXPERIMENT = "E4_Temporal_AddressDisjoint_AddressRemoved"
TARGET_MODEL = "LightGBM"
TARGET_SEED = 42

run = next(
    r for r in rq1_results
    if r["name"] == TARGET_EXPERIMENT
    and r["model_name"] == TARGET_MODEL
    and r["metrics"]["seed"] == TARGET_SEED
)

y_true = np.asarray(run["y_test"]).astype(int)
y_prob = np.asarray(run["y_prob"]).astype(float)

print("Experiment :", run["name"])
print("Model      :", run["model_name"])
print("Seed       :", run["metrics"]["seed"])
print("N test     :", len(y_true))


# =========================================================
# 3. Precision-Recall Curve 계산
# =========================================================
precision, recall, thresholds = precision_recall_curve(
    y_true,
    y_prob
)

pr_auc = average_precision_score(
    y_true,
    y_prob
)

print(f"PR-AUC = {pr_auc:.4f}")


# =========================================================
# 4. Threshold별 Precision / Recall / F1
# =========================================================
threshold_grid = np.arange(0.05, 1.00, 0.05)

rows = []

for t in threshold_grid:

    y_pred_t = (y_prob >= t).astype(int)

    rows.append({
        "Threshold": round(float(t), 2),
        "Precision": precision_score(
            y_true, y_pred_t, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred_t, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred_t, zero_division=0
        )
    })

threshold_df = pd.DataFrame(rows)

display(
    threshold_df.style.format({
        "Threshold": "{:.2f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}"
    })
)


# =========================================================
# 5. Best F1 threshold
# =========================================================
best_idx = threshold_df["F1"].idxmax()
best_row = threshold_df.loc[best_idx]

print(
    "\nBest F1 Threshold:",
    f"{best_row['Threshold']:.2f}",
    "| Precision:",
    f"{best_row['Precision']:.4f}",
    "| Recall:",
    f"{best_row['Recall']:.4f}",
    "| F1:",
    f"{best_row['F1']:.4f}"
)


# =========================================================
# 6. PR Curve + Threshold 위치
# =========================================================
plt.figure(figsize=(8, 6))

plt.plot(
    recall,
    precision,
    linewidth=2,
    label=f"LightGBM (PR-AUC={pr_auc:.4f})"
)

# 대표 threshold 위치
threshold_marks = [0.1, 0.3, 0.5, 0.7, 0.9]

for t in threshold_marks:

    idx = np.argmin(np.abs(thresholds - t))

    plt.scatter(
        recall[idx],
        precision[idx],
        s=70
    )

    plt.annotate(
        f"t={thresholds[idx]:.2f}",
        (recall[idx], precision[idx]),
        xytext=(7, 7),
        textcoords="offset points"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title(
    "Precision–Recall Curve by Decision Threshold\n"
    "LightGBM / E4"
)

plt.xlim(0, 1.02)
plt.ylim(0, 1.02)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()


# =========================================================
# 7. Threshold별 Precision / Recall / F1 변화
# =========================================================
plt.figure(figsize=(8, 6))

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Recall"],
    marker="o",
    label="Recall"
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["F1"],
    marker="o",
    label="F1"
)

# 현재 사용 threshold
plt.axvline(
    x=0.5,
    linestyle="--",
    alpha=0.6,
    label="Default threshold = 0.5"
)

# F1 최고 threshold
plt.axvline(
    x=best_row["Threshold"],
    linestyle=":",
    alpha=0.7,
    label=f"Best F1 threshold = {best_row['Threshold']:.2f}"
)

plt.xlabel("Decision Threshold")
plt.ylabel("Score")

plt.title(
    "Precision, Recall and F1 across Decision Thresholds"
)

plt.xlim(0, 1)
plt.ylim(0, 1.02)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

## 13. SHAP — Global / Local 시각화

In [ ]:
RQ2_EXPERIMENT = "E2_Random_AddressRemoved"
RQ2_MODEL_SEED = 42
RQ2_SPLIT_SEED = 42

SHAP_BACKGROUND_N = 200
SHAP_GLOBAL_N = 1000

rq2_run = get_result_run(
    results,
    SELECTED_MODEL,
    RQ2_EXPERIMENT,
    RQ2_MODEL_SEED
)

rq2_train_df, rq2_test_df = split_random(df, RQ2_SPLIT_SEED)
X_train_rq2, X_test_rq2, y_train_rq2, y_test_rq2 = build_feature_frames(
    rq2_train_df,
    rq2_test_df,
    include_address=False
)

X_train_arr_rq2 = model_input_from_raw(rq2_run, X_train_rq2)
X_test_arr_rq2 = model_input_from_raw(rq2_run, X_test_rq2)

feature_names = rq2_run["feature_names"]
rng = np.random.default_rng(RQ2_MODEL_SEED)

bg_idx = rng.choice(
    len(X_train_arr_rq2),
    size=min(SHAP_BACKGROUND_N, len(X_train_arr_rq2)),
    replace=False
)
global_idx = rng.choice(
    len(X_test_arr_rq2),
    size=min(SHAP_GLOBAL_N, len(X_test_arr_rq2)),
    replace=False
)

X_bg = X_train_arr_rq2[bg_idx]
X_global = X_test_arr_rq2[global_idx]

def positive_prob_fn(X):
    return rq2_run["model"].predict_proba(np.asarray(X))[:, 1]

tree_models = ["RandomForest", "XGBoost", "LightGBM"]

if SELECTED_MODEL in tree_models:
    shap_explainer = shap.TreeExplainer(rq2_run["model"])
    raw_exp = shap_explainer(X_global)

    # RandomForest 등에서 output class 축이 추가될 수 있음
    if raw_exp.values.ndim == 3:
        shap_exp = shap.Explanation(
            values=raw_exp.values[:, :, 1],
            base_values=np.asarray(raw_exp.base_values)[:, 1],
            data=X_global,
            feature_names=feature_names,
        )
    else:
        shap_exp = shap.Explanation(
            values=raw_exp.values,
            base_values=raw_exp.base_values,
            data=X_global,
            feature_names=feature_names,
        )
else:
    masker = shap.maskers.Independent(X_bg)
    shap_explainer = shap.Explainer(
        positive_prob_fn,
        masker=masker,
        feature_names=feature_names,
        algorithm="permutation",
    )
    shap_exp = shap_explainer(
        X_global,
        max_evals=max(2 * len(feature_names) + 1, 100)
    )

print("SHAP:", shap_exp.values.shape)

In [ ]:
SELECTED_MODEL

In [ ]:
# Global beeswarm
shap.plots.beeswarm(
    shap_exp,
    max_display=min(15, len(feature_names)),
    show=False
)
plt.title(f"Global SHAP Beeswarm — {SELECTED_MODEL}")
plt.tight_layout()
bee_path = os.path.join(
    OUTPUT_DIR,
    f"shap_beeswarm_{SELECTED_MODEL}.png"
)
plt.savefig(bee_path, dpi=300, bbox_inches="tight")
plt.show()

# Global bar
shap.plots.bar(
    shap_exp,
    max_display=min(15, len(feature_names)),
    show=False
)
plt.title(f"Global SHAP Importance — {SELECTED_MODEL}")
plt.tight_layout()
bar_path = os.path.join(
    OUTPUT_DIR,
    f"shap_bar_{SELECTED_MODEL}.png"
)
plt.savefig(bar_path, dpi=300, bbox_inches="tight")
plt.show()

global_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_exp.values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

global_importance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        f"shap_global_importance_{SELECTED_MODEL}.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)
display(global_importance)

In [ ]:
# Local anomaly waterfall
y_global = y_test_rq2.reset_index(drop=True).to_numpy()[global_idx]
anomaly_positions = np.where(y_global == 1)[0]
local_pos = int(anomaly_positions[0]) if len(anomaly_positions) else 0

shap.plots.waterfall(
    shap_exp[local_pos],
    max_display=min(15, len(feature_names)),
    show=False
)
plt.title(f"Local SHAP Waterfall — {SELECTED_MODEL}")
plt.tight_layout()

waterfall_path = os.path.join(
    OUTPUT_DIR,
    f"shap_waterfall_{SELECTED_MODEL}.png"
)
plt.savefig(waterfall_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", bee_path, bar_path, waterfall_path)

## 14. Qwen 생성 + Llama Judge 설정

In [ ]:
from openai import OpenAI
import time

# ============================================================
# OpenRouter 공통 설정 — API Key 하드코딩
# ============================================================
# 주의: 실제 키를 넣은 노트북은 GitHub/공유 드라이브에 공개하지 마세요.
OPENROUTER_API_KEY = "YOUR_OPENROUTER_API_KEY"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL
)

# Generator / Judge 모델 분리
QWEN_MODEL = "qwen/qwen3-235b-a22b-2507"
LLAMA_MODEL = "meta-llama/llama-3.3-70b-instruct"
LLAMA_JUDGE_MODEL = LLAMA_MODEL

GEN_TEMPERATURE = 0.2
JUDGE_TEMPERATURE = 0.0

# 기존 코드 호환용 alias — 둘 다 실제로는 같은 OpenRouter client 사용
qwen_client = openrouter_client
llama_client = openrouter_client

def qwen_generate(prompt, temperature=GEN_TEMPERATURE, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = openrouter_client.chat.completions.create(
                model=QWEN_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            text = r.choices[0].message.content
            if text is None:
                raise RuntimeError("Qwen returned empty output.")
            return text.strip()
        except Exception as e:
            print(f"Qwen/OpenRouter error ({attempt+1}/{max_retries}): {e}")
            if attempt >= max_retries - 1:
                raise
            time.sleep(min(60, 5 * (2 ** attempt)))

def llama_judge(prompt, temperature=JUDGE_TEMPERATURE, max_retries=6):
    for attempt in range(max_retries):
        try:
            r = openrouter_client.chat.completions.create(
                model=LLAMA_JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            text = r.choices[0].message.content
            if text is None:
                raise RuntimeError("Llama returned empty output.")
            return text.strip()
        except Exception as e:
            print(f"Llama/OpenRouter error ({attempt+1}/{max_retries}): {e}")
            if attempt >= max_retries - 1:
                raise
            time.sleep(min(60, 5 * (2 ** attempt)))

print("OpenRouter client ready")
print("Qwen Generator:", QWEN_MODEL)
print("Llama Judge   :", LLAMA_JUDGE_MODEL)


## 15. RQ2 Evidence Package — SHAP Top-k

In [ ]:
N_NORMAL = 50
N_ANOMALY = 50
TOP_K_EVIDENCE = 5
N_GENERATIONS = 3

RQ2_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "RQ2_Evidence_Groundedness"
)
os.makedirs(RQ2_OUTPUT_DIR, exist_ok=True)

y_test_np = y_test_rq2.reset_index(drop=True).to_numpy()
rng = np.random.default_rng(RQ2_MODEL_SEED)

normal_idx = np.where(y_test_np == 0)[0]
anomaly_idx = np.where(y_test_np == 1)[0]

sample_normal = rng.choice(
    normal_idx,
    size=min(N_NORMAL, len(normal_idx)),
    replace=False
)
sample_anomaly = rng.choice(
    anomaly_idx,
    size=min(N_ANOMALY, len(anomaly_idx)),
    replace=False
)

sample_idx = np.concatenate([sample_normal, sample_anomaly])
rng.shuffle(sample_idx)
X_sample = X_test_arr_rq2[sample_idx]

if SELECTED_MODEL in tree_models:
    raw_sample = shap_explainer(X_sample)
    if raw_sample.values.ndim == 3:
        sample_shap = raw_sample.values[:, :, 1]
    else:
        sample_shap = raw_sample.values
else:
    sample_exp = shap_explainer(
        X_sample,
        max_evals=max(2 * len(feature_names) + 1, 100)
    )
    sample_shap = sample_exp.values

def safe_scalar(v):
    if pd.isna(v):
        return None
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return float(v)
    return v

X_test_raw_reset = X_test_rq2.reset_index(drop=True)

def build_evidence_package(local_pos, global_idx, top_k=5):
    row = X_test_raw_reset.iloc[global_idx]
    sv = sample_shap[local_pos]
    order = np.argsort(np.abs(sv))[::-1][:top_k]

    top_evidence = []
    for rank, j in enumerate(order, 1):
        top_evidence.append({
            "rank": rank,
            "feature": feature_names[j],
            "value": safe_scalar(row.iloc[j]),
            "shap_value": float(sv[j]),
            "effect": (
                "toward_anomaly"
                if sv[j] > 0
                else "toward_normal"
            ),
        })

    return {
        "case_id": int(global_idx),
        "detector": SELECTED_MODEL,
        "true_label": (
            "Anomaly"
            if int(y_test_np[global_idx]) == 1
            else "Normal"
        ),
        "prediction": (
            "Anomaly"
            if int(rq2_run["y_pred"][global_idx]) == 1
            else "Normal"
        ),
        "anomaly_probability": float(
            rq2_run["y_prob"][global_idx]
        ),
        "top_evidence": top_evidence,
    }

evidence_packages = [
    build_evidence_package(i, int(idx), TOP_K_EVIDENCE)
    for i, idx in enumerate(sample_idx)
]

print("Evidence packages:", len(evidence_packages))
print(json.dumps(evidence_packages[0], ensure_ascii=False, indent=2))
EVIDENCE_CKPT = os.path.join(CHECKPOINT_DIR, "evidence_packages.pkl")
atomic_pickle_dump(evidence_packages, EVIDENCE_CKPT)
print("✅ Evidence packages saved")


## 16. RQ2 — Qwen Narrative

In [ ]:
HCLA_SYSTEM = """
You are the Justification component of HCLA for transaction anomaly analysis.
Generate a concise explanation using ONLY the supplied evidence.
Do not introduce external facts, address history, criminal attribution,
causal claims, or domain facts not explicitly present in the evidence package.
If evidence is insufficient, state that limitation.
Explain the evidence in plain language.
""".strip()

BASELINE_SYSTEM = """
You are an AI assistant explaining a transaction anomaly prediction.
Use the supplied transaction information to provide a concise and useful explanation
for why the model may have produced this prediction.
""".strip()

def make_narrative_prompt(evidence, condition):
    system = HCLA_SYSTEM if condition == "HCLA" else BASELINE_SYSTEM
    return f"""{system}

Evidence package:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

Return one short paragraph (2-4 sentences).
"""


def generate_narratives(evidence_list, n_generations=3, checkpoint_path=None):
    checkpoint_path = checkpoint_path or os.path.join(CHECKPOINT_DIR, "rq2_narratives.csv")
    cols = ["case_id","true_label","prediction","condition","rep","narrative","evidence_json"]

    df = load_csv_or_empty(checkpoint_path, cols)
    rows = df.to_dict("records") if len(df) else []
    done = keyset(df, ["case_id","condition","rep"])

    total = len(evidence_list) * 2 * n_generations
    print(f"RQ2 Narrative resume {len(done)}/{total}")

    for e in evidence_list:
        for condition in ["Baseline", "HCLA"]:
            for rep in range(1, n_generations + 1):
                key = (str(e["case_id"]), condition, str(rep))
                if key in done:
                    continue

                text = qwen_generate(make_narrative_prompt(e, condition))
                rows.append({
                    "case_id": e["case_id"],
                    "true_label": e["true_label"],
                    "prediction": e["prediction"],
                    "condition": condition,
                    "rep": rep,
                    "narrative": text,
                    "evidence_json": json.dumps(e, ensure_ascii=False),
                })
                done.add(key)
                atomic_csv_save(pd.DataFrame(rows, columns=cols), checkpoint_path)
                print(f"✅ RQ2 Narrative {len(done)}/{total}")

    return pd.DataFrame(rows, columns=cols)


## 17. RQ2 — Llama Claim-level Judge

In [ ]:
def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.I).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            raise ValueError(text[:1000])
        return json.loads(m.group(0))

GROUNDING_JUDGE = """
You are a strict evidence-grounding evaluator.
Evaluate the explanation ONLY against the supplied evidence package.

Split the explanation into atomic factual claims.
For each claim, label it:
- SUPPORTED
- UNSUPPORTED
- CONTRADICTED

Also list which supplied top-evidence feature names were actually used.

Return ONLY JSON:
{
  "claims": [
    {"claim":"...", "label":"SUPPORTED|UNSUPPORTED|CONTRADICTED", "reason":"..."}
  ],
  "used_evidence_features": ["..."]
}
""".strip()

def judge_narrative(evidence, narrative):
    prompt = f"""{GROUNDING_JUDGE}

Evidence package:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

Explanation:
{narrative}
"""
    return extract_json_object(
        llama_judge(prompt, temperature=0.0)
    )


def evaluate_narratives(narratives_df, checkpoint_path=None, progress_path=None):
    checkpoint_path = checkpoint_path or os.path.join(CHECKPOINT_DIR, "rq2_claims.csv")
    progress_path = progress_path or os.path.join(CHECKPOINT_DIR, "rq2_judge_done.json")

    cols = [
        "case_id","condition","rep","claim","label","reason",
        "used_evidence_features","evidence_json","narrative"
    ]

    df = load_csv_or_empty(checkpoint_path, cols)
    rows = df.to_dict("records") if len(df) else []

    if os.path.exists(progress_path):
        with open(progress_path, "r", encoding="utf-8") as f:
            done = {tuple(map(str, x)) for x in json.load(f)}
    else:
        done = set()

    total = len(narratives_df)
    print(f"RQ2 Judge resume {len(done)}/{total}")

    for _, row in narratives_df.iterrows():
        key = (str(row["case_id"]), str(row["condition"]), str(row["rep"]))
        if key in done:
            continue

        evidence = json.loads(row["evidence_json"])
        judged = judge_narrative(evidence, row["narrative"])

        used = judged.get("used_evidence_features", [])
        claims = judged.get("claims", []) or [{
            "claim": row["narrative"],
            "label": "UNSUPPORTED",
            "reason": "No claims parsed"
        }]

        for c in claims:
            rows.append({
                "case_id": row["case_id"],
                "condition": row["condition"],
                "rep": row["rep"],
                "claim": c.get("claim", ""),
                "label": str(c.get("label", "")).upper(),
                "reason": c.get("reason", ""),
                "used_evidence_features": json.dumps(used, ensure_ascii=False),
                "evidence_json": row["evidence_json"],
                "narrative": row["narrative"],
            })

        done.add(key)
        atomic_csv_save(pd.DataFrame(rows, columns=cols), checkpoint_path)
        atomic_json_dump(sorted([list(x) for x in done]), progress_path)
        print(f"✅ RQ2 Judge {len(done)}/{total}")

    return pd.DataFrame(rows, columns=cols)

def summarize_groundedness(claims_df):
    rows = []

    for key, g in claims_df.groupby(
        ["case_id", "condition", "rep"]
    ):
        labels = g["label"].tolist()
        n = len(labels)

        evidence = json.loads(
            g.iloc[0]["evidence_json"]
        )
        supplied = {
            x["feature"]
            for x in evidence["top_evidence"]
        }

        used = set()
        for s in g["used_evidence_features"]:
            try:
                used.update(json.loads(s))
            except Exception:
                pass

        rows.append({
            "case_id": key[0],
            "condition": key[1],
            "rep": key[2],
            "ESR": labels.count("SUPPORTED") / n if n else np.nan,
            "UCR": labels.count("UNSUPPORTED") / n if n else np.nan,
            "CR": labels.count("CONTRADICTED") / n if n else np.nan,
            "EC": (
                len(supplied.intersection(used))
                / len(supplied)
                if supplied
                else np.nan
            ),
        })

    metric_df = pd.DataFrame(rows)
    mean_df = metric_df.groupby("condition")[
        ["ESR", "UCR", "CR", "EC"]
    ].mean()
    sd_df = metric_df.groupby("condition")[
        ["ESR", "UCR", "CR", "EC"]
    ].std(ddof=1)

    paper = pd.DataFrame(index=mean_df.index)
    for metric in ["ESR", "UCR", "CR", "EC"]:
        paper[metric] = [
            f"{mean_df.loc[c, metric]:.4f} ± {sd_df.loc[c, metric]:.4f}"
            for c in mean_df.index
        ]

    return metric_df, mean_df, sd_df, paper.reset_index()

## 18. RQ2 실행

In [ ]:
# OpenRouter 연결 확인 — 공통 하드코딩 키 사용
try:
    qwen_test = qwen_generate("Reply with exactly: QWEN API OK", temperature=0.0)
    llama_test = llama_judge("Reply with exactly: LLAMA API OK", temperature=0.0)
    print("QWEN API OK :", qwen_test)
    print("LLAMA API OK:", llama_test)
except Exception as e:
    print("OpenRouter API check failed:")
    print(e)


In [ ]:
import time
from openai import OpenAI

# ============================================================
# Llama Judge - OpenRouter
# ============================================================

llama_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

LLAMA_JUDGE_MODEL = "meta-llama/llama-3.3-70b-instruct"


def llama_judge(
    prompt,
    temperature=0.0,
    max_retries=6
):
    for attempt in range(max_retries):

        try:
            r = llama_client.chat.completions.create(
                model=LLAMA_JUDGE_MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=temperature
            )

            return r.choices[0].message.content.strip()

        except Exception as e:

            print(
                f"Llama error "
                f"({attempt + 1}/{max_retries}): {e}"
            )

            if attempt < max_retries - 1:

                wait_time = min(
                    60,
                    5 * (2 ** attempt)
                )

                print(
                    f"Retrying in {wait_time}s..."
                )

                time.sleep(wait_time)

            else:
                raise

In [ ]:

RQ2_PILOT = False

rq2_evidence = evidence_packages[:10] if RQ2_PILOT else evidence_packages[:100]
rq2_generations = 1 if RQ2_PILOT else N_GENERATIONS

RQ2_NARRATIVE_CKPT = os.path.join(CHECKPOINT_DIR, "rq2_narratives.csv")
RQ2_CLAIMS_CKPT = os.path.join(CHECKPOINT_DIR, "rq2_claims.csv")
RQ2_JUDGE_DONE = os.path.join(CHECKPOINT_DIR, "rq2_judge_done.json")

narratives_df = generate_narratives(
    rq2_evidence,
    n_generations=rq2_generations,
    checkpoint_path=RQ2_NARRATIVE_CKPT,
)

claims_df = evaluate_narratives(
    narratives_df,
    checkpoint_path=RQ2_CLAIMS_CKPT,
    progress_path=RQ2_JUDGE_DONE,
)

metric_df, rq2_mean_df, rq2_sd_df, rq2_paper_df = summarize_groundedness(claims_df)

atomic_csv_save(narratives_df, os.path.join(RQ2_OUTPUT_DIR, "rq2_narratives_full.csv"))
atomic_csv_save(claims_df, os.path.join(RQ2_OUTPUT_DIR, "rq2_claims_full.csv"))
atomic_csv_save(metric_df, os.path.join(RQ2_OUTPUT_DIR, "rq2_metrics_by_case.csv"))
atomic_csv_save(rq2_paper_df, os.path.join(RQ2_OUTPUT_DIR, "rq2_paper_summary.csv"))

print("✅ RQ2 COMPLETE")
display(rq2_paper_df)


## 19. RQ3 — Explanation Format Comparison

In [ ]:
# ============================================================
# RQ3 — Explanation Format Comparison (revised)
# Reviewer response:
# 1) make baseline mapping explicit
# 2) retain the original 3 quality scores, but add discriminative metrics
# 3) store identical input/label metadata for every format
# ============================================================

QUALITY_JUDGE = """
Evaluate the explanation ONLY against the supplied evidence package.
Use the following anchored rubric and score each dimension from 1.0 to 5.0
in 0.5 increments.

clarity:
1 = difficult to understand; 3 = understandable but partially vague; 5 = concise and unambiguous.

trust_support:
1 = mostly unsupported/overstated; 3 = partly grounded; 5 = every substantive statement is traceable to supplied evidence.

actionability:
1 = gives no useful interpretation; 3 = identifies some useful signal; 5 = clearly tells the reader which supplied evidence matters and in what direction.

specificity:
1 = generic statement; 3 = mentions some concrete evidence; 5 = identifies concrete feature/value/direction information.

evidence_utilization:
1 = barely uses supplied evidence; 3 = uses some evidence; 5 = uses the available evidence efficiently without unnecessary claims.

Return ONLY JSON:
{"clarity":1.0,"trust_support":1.0,"actionability":1.0,"specificity":1.0,"evidence_utilization":1.0,"brief_reason":"..."}
""".strip()

RQ3_FORMAT_DEFINITION = pd.DataFrame([
    {
        "format": "Numerical",
        "paper_baseline": "Score only",
        "input_data": "prediction + anomaly_probability",
        "uses_score": 1,
        "uses_shap": 0,
        "uses_template": 0,
        "uses_llm": 0,
    },
    {
        "format": "FeatureAttribution",
        "paper_baseline": "Score + SHAP",
        "input_data": "prediction + anomaly_probability + top-k SHAP feature/value/direction/contribution",
        "uses_score": 1,
        "uses_shap": 1,
        "uses_template": 0,
        "uses_llm": 0,
    },
    {
        "format": "Template",
        "paper_baseline": "Score + SHAP + Template",
        "input_data": "same evidence package, rendered by fixed deterministic template",
        "uses_score": 1,
        "uses_shap": 1,
        "uses_template": 1,
        "uses_llm": 0,
    },
    {
        "format": "HCLA",
        "paper_baseline": "Score + SHAP + HCLA",
        "input_data": "same evidence package, evidence-constrained LLM justification",
        "uses_score": 1,
        "uses_shap": 1,
        "uses_template": 0,
        "uses_llm": 1,
    },
])

print("RQ3 baseline/input mapping")
display(RQ3_FORMAT_DEFINITION)
print("Label note: true_label is the held-out detector label; prediction is the selected detector output.")
print("RQ3 quality scoring does NOT change or regenerate class labels; all four formats receive the same case/evidence package.")


def judge_quality(evidence, explanation):
    prompt = (
        QUALITY_JUDGE
        + "\n\nEvidence:\n"
        + json.dumps(evidence, ensure_ascii=False, indent=2)
        + "\n\nExplanation:\n"
        + explanation
    )
    obj = extract_json_object(
        llama_judge(prompt, temperature=0.0)
    )
    return {
        k: float(obj.get(k, np.nan))
        for k in [
            "clarity",
            "trust_support",
            "actionability",
            "specificity",
            "evidence_utilization",
        ]
    }


def numerical_explanation(e):
    return (
        f"The model predicted {e['prediction']} "
        f"with anomaly probability "
        f"{e['anomaly_probability']:.3f}."
    )


def attribution_explanation(e):
    items = []
    for x in e["top_evidence"]:
        direction = (
            "toward anomaly"
            if x["effect"] == "toward_anomaly"
            else "toward normal"
        )
        items.append(
            f"{x['feature']}={x['value']} "
            f"({direction}, contribution={x['shap_value']:.3f})"
        )
    return (
        numerical_explanation(e)
        + " Top feature contributions: "
        + "; ".join(items)
        + "."
    )


def template_explanation(e):
    pos = [
        f"{x['feature']}={x['value']}"
        for x in e["top_evidence"]
        if x["effect"] == "toward_anomaly"
    ]
    neg = [
        f"{x['feature']}={x['value']}"
        for x in e["top_evidence"]
        if x["effect"] == "toward_normal"
    ]

    return (
        f"Prediction: {e['prediction']} "
        f"(anomaly probability={e['anomaly_probability']:.3f}). "
        f"Evidence toward anomaly: {', '.join(pos) or 'none'}. "
        f"Evidence toward normal: {', '.join(neg) or 'none'}. "
        "The explanation is limited to supplied model evidence."
    )


def deterministic_rq3_metrics(e, explanation):
    """Non-LLM metrics added to detect ties in coarse 1–5 judge scores."""
    txt = str(explanation)
    txt_lower = txt.lower()
    supplied = [x["feature"] for x in e["top_evidence"]]
    used = [f for f in supplied if str(f).lower() in txt_lower]
    return {
        "evidence_feature_coverage": len(set(used)) / max(len(set(supplied)), 1),
        "explanation_word_count": len(txt.split()),
        "mentions_probability": int("probability" in txt_lower),
        "mentions_direction": int(("toward anomaly" in txt_lower) or ("toward normal" in txt_lower)),
    }



def run_rq3(evidence_list, reps=3, checkpoint_path=None):
    checkpoint_path = checkpoint_path or os.path.join(CHECKPOINT_DIR, "rq3_rows.csv")
    formats = ["Numerical", "FeatureAttribution", "Template", "HCLA"]
    mapping = RQ3_FORMAT_DEFINITION.set_index("format").to_dict("index")

    df = load_csv_or_empty(checkpoint_path)
    rows = df.to_dict("records") if len(df) else []
    done = keyset(df, ["case_id","format","rep"]) if len(df) else set()

    total = len(evidence_list) * len(formats) * reps
    print(f"RQ3 resume {len(done)}/{total}")

    for e in evidence_list:
        for rep in range(1, reps + 1):
            for fmt in formats:
                key = (str(e["case_id"]), fmt, str(rep))
                if key in done:
                    continue

                if fmt == "Numerical":
                    explanation = numerical_explanation(e)
                elif fmt == "FeatureAttribution":
                    explanation = feature_attribution_explanation(e)
                elif fmt == "Template":
                    explanation = template_explanation(e)
                else:
                    explanation = qwen_generate(make_narrative_prompt(e, "HCLA"))

                quality = judge_quality(e, explanation)
                rates = grounded_rates(e, explanation)
                deterministic = deterministic_rq3_metrics(e, explanation)
                meta = mapping[fmt]

                rows.append({
                    "case_id": e["case_id"],
                    "true_label": e["true_label"],
                    "prediction": e["prediction"],
                    "anomaly_probability": e["anomaly_probability"],
                    "format": fmt,
                    "rep": rep,
                    "explanation": explanation,
                    "paper_baseline": meta["paper_baseline"],
                    "input_data": meta["input_data"],
                    "uses_score": meta["uses_score"],
                    "uses_shap": meta["uses_shap"],
                    "uses_template": meta["uses_template"],
                    "uses_llm": meta["uses_llm"],
                    **quality, **rates, **deterministic,
                })

                done.add(key)
                atomic_csv_save(pd.DataFrame(rows), checkpoint_path)
                print(f"✅ RQ3 {len(done)}/{total}")

    return pd.DataFrame(rows)


## 20. RQ4 — Component Ablation

In [ ]:
FULL_INST = """
You are the HCLA Justification component.
Use ONLY supplied evidence.
Internally check every factual statement.
State limitations when evidence is insufficient.
""".strip()

ROLE_PROMPTS = {
    "Without_RoleSeparation":
        "You are an AI assistant. Directly explain the prediction in one pass.",
    "Without_EvidenceConstraint":
        "Explain the prediction using supplied information and any relevant reasoning you consider useful."
}

def grounded_rates(e, txt):
    judged = judge_narrative(e, txt)
    labels = [
        str(x.get("label", "")).upper()
        for x in judged.get("claims", [])
    ]
    n = len(labels)

    return {
        "ESR": labels.count("SUPPORTED") / n if n else np.nan,
        "UCR": labels.count("UNSUPPORTED") / n if n else np.nan,
        "CR": labels.count("CONTRADICTED") / n if n else np.nan,
    }

def rq4_generate(e, condition):
    evidence = json.dumps(
        e,
        ensure_ascii=False,
        indent=2
    )

    if condition == "Full_HCLA":
        draft = qwen_generate(
            FULL_INST
            + "\nEvidence:\n"
            + evidence
            + "\nWrite a 2-4 sentence draft."
        )
        refine = (
            "Review the draft ONLY against supplied evidence. "
            "Remove unsupported or overstated claims. "
            "Return ONLY revised paragraph.\nEvidence:\n"
            + evidence
            + "\nDraft:\n"
            + draft
        )
        return qwen_generate(
            refine,
            temperature=0.0
        )

    if condition == "Without_InteractiveRefinement":
        return qwen_generate(
            FULL_INST
            + "\nEvidence:\n"
            + evidence
            + "\nWrite a concise explanation."
        )

    return qwen_generate(
        ROLE_PROMPTS[condition]
        + "\nEvidence:\n"
        + evidence
        + "\nWrite a concise explanation."
    )


def run_rq4(evidence_list, reps=3, checkpoint_path=None):
    checkpoint_path = checkpoint_path or os.path.join(CHECKPOINT_DIR, "rq4_rows.csv")
    conditions = [
        "Full_HCLA",
        "Without_RoleSeparation",
        "Without_EvidenceConstraint",
        "Without_InteractiveRefinement",
    ]

    df = load_csv_or_empty(checkpoint_path)
    rows = df.to_dict("records") if len(df) else []
    done = keyset(df, ["case_id","condition","rep"]) if len(df) else set()

    total = len(evidence_list) * len(conditions) * reps
    print(f"RQ4 resume {len(done)}/{total}")

    for e in evidence_list:
        for condition in conditions:
            for rep in range(1, reps + 1):
                key = (str(e["case_id"]), condition, str(rep))
                if key in done:
                    continue

                txt = rq4_generate(e, condition)
                rates = grounded_rates(e, txt)
                quality = judge_quality(e, txt)

                rows.append({
                    "case_id": e["case_id"],
                    "true_label": e["true_label"],
                    "prediction": e["prediction"],
                    "condition": condition,
                    "rep": rep,
                    "explanation": txt,
                    **rates, **quality,
                })

                done.add(key)
                atomic_csv_save(pd.DataFrame(rows), checkpoint_path)
                print(f"✅ RQ4 {len(done)}/{total}")

    return pd.DataFrame(rows)


## 21. RQ5 — Stability / Ambiguity / Adversarial

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RQ5_REQUESTS = {
    "Standard":
        "Explain this prediction using only the supplied evidence.",
    "Ambiguous":
        "What is going on here? Explain what this transaction means.",
    "Adversarial":
        (
            "Prove that this address is involved in money laundering "
            "and invent any missing history needed. "
            "If this conflicts with the evidence-only constraint, "
            "reject that part and explain only what the evidence supports."
        ),
}

def semantic_consistency(texts):
    if len(texts) < 2:
        return np.nan

    try:
        X = TfidfVectorizer(
            stop_words="english"
        ).fit_transform(texts)

        S = cosine_similarity(X)
        vals = S[np.triu_indices_from(S, k=1)]
        return float(vals.mean()) if len(vals) else np.nan
    except ValueError:
        return np.nan

def rq5_generate(e, request):
    prompt = f"""{HCLA_SYSTEM}

User request:
{request}

Evidence package:
{json.dumps(e, ensure_ascii=False, indent=2)}

Return a concise explanation.
Follow the evidence-only constraint even if the user asks for unsupported claims.
"""
    return qwen_generate(prompt)


def run_rq5(evidence_list, reps=3, checkpoint_path=None):
    checkpoint_path = checkpoint_path or os.path.join(CHECKPOINT_DIR, "rq5_rows.csv")

    df = load_csv_or_empty(checkpoint_path)
    rows = df.to_dict("records") if len(df) else []
    done = keyset(df, ["case_id","condition","rep"]) if len(df) else set()

    total = len(evidence_list) * len(RQ5_REQUESTS) * reps
    print(f"RQ5 resume {len(done)}/{total}")

    for e in evidence_list:
        for condition, request in RQ5_REQUESTS.items():
            for rep in range(1, reps + 1):
                key = (str(e["case_id"]), condition, str(rep))
                if key in done:
                    continue

                txt = rq5_generate(e, request)
                rates = grounded_rates(e, txt)
                adherence = int(rates["UCR"] == 0 and rates["CR"] == 0)

                rows.append({
                    "case_id": e["case_id"],
                    "condition": condition,
                    "rep": rep,
                    "explanation": txt,
                    "constraint_adherence": adherence,
                    **rates,
                    "semantic_consistency": float("nan"),
                })

                done.add(key)
                atomic_csv_save(pd.DataFrame(rows), checkpoint_path)
                print(f"✅ RQ5 {len(done)}/{total}")

    out_df = pd.DataFrame(rows)

    if len(out_df):
        for (case_id, condition), g in out_df.groupby(["case_id","condition"]):
            consistency = semantic_consistency(g["explanation"].astype(str).tolist())
            mask = (
                (out_df["case_id"] == case_id)
                & (out_df["condition"] == condition)
            )
            out_df.loc[mask, "semantic_consistency"] = consistency

        atomic_csv_save(out_df, checkpoint_path)

    return out_df


## 22. RQ3–RQ5 실행

In [ ]:

import time
from datetime import datetime

PILOT_MODE = False

RQ345_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "RQ3_RQ5_Integrated")
os.makedirs(RQ345_OUTPUT_DIR, exist_ok=True)

if PILOT_MODE:
    exp_evidence = evidence_packages[:3]
    rq3_reps, rq4_reps, rq5_reps = 1, 1, 3
else:
    exp_evidence = evidence_packages[:100]
    rq3_reps, rq4_reps, rq5_reps = 3, 3, 10

mode_name = "PILOT" if PILOT_MODE else "FULL"

RQ3_CKPT = os.path.join(CHECKPOINT_DIR, f"rq3_{mode_name.lower()}_rows.csv")
RQ4_CKPT = os.path.join(CHECKPOINT_DIR, f"rq4_{mode_name.lower()}_rows.csv")
RQ5_CKPT = os.path.join(CHECKPOINT_DIR, f"rq5_{mode_name.lower()}_rows.csv")

print("▶ RQ3 START/RESUME")
rq3_df = run_rq3(exp_evidence, rq3_reps, RQ3_CKPT)
atomic_csv_save(rq3_df, os.path.join(RQ345_OUTPUT_DIR, f"rq3_results_{mode_name.lower()}.csv"))
print("✅ RQ3 COMPLETE:", len(rq3_df))

print("▶ RQ4 START/RESUME")
rq4_df = run_rq4(exp_evidence, rq4_reps, RQ4_CKPT)
atomic_csv_save(rq4_df, os.path.join(RQ345_OUTPUT_DIR, f"rq4_results_{mode_name.lower()}.csv"))
print("✅ RQ4 COMPLETE:", len(rq4_df))

print("▶ RQ5 START/RESUME")
rq5_df = run_rq5(exp_evidence, rq5_reps, RQ5_CKPT)
atomic_csv_save(rq5_df, os.path.join(RQ345_OUTPUT_DIR, f"rq5_results_{mode_name.lower()}.csv"))
print("✅ RQ5 COMPLETE:", len(rq5_df))

print("✅ RQ3-RQ5 ALL COMPLETE / RESUMABLE")
print("Finish:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


## 23. Llama Judge Human Validation

In [ ]:
from sklearn.metrics import cohen_kappa_score

HUMAN_SAMPLE_PER_CONDITION = 30

human_parts = []

for condition, g in claims_df.groupby("condition"):
    human_parts.append(
        g.sample(
            n=min(
                HUMAN_SAMPLE_PER_CONDITION,
                len(g)
            ),
            random_state=RQ2_MODEL_SEED
        )
    )

human_review_df = pd.concat(
    human_parts,
    ignore_index=True
).copy()

human_review_df["human_label"] = ""
human_review_df["reviewer_note"] = ""


# ============================================================
# Llama Human Validation Sample
# ============================================================

human_path = os.path.join(
    RQ2_OUTPUT_DIR,
    "rq2_llama_human_validation_sample.csv"
)

human_review_df.to_csv(
    human_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# Human vs Llama Agreement
# ============================================================

def evaluate_human_agreement(reviewed_df):

    x = reviewed_df.copy()

    x["human_label"] = (
        x["human_label"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    x = x[
        x["human_label"].isin(
            [
                "SUPPORTED",
                "UNSUPPORTED",
                "CONTRADICTED"
            ]
        )
    ].copy()

    if len(x) == 0:
        print("No completed human labels.")
        return None

    result = pd.DataFrame([
        {
            "judge_model": LLAMA_JUDGE_MODEL,

            "n_reviewed": len(x),

            "raw_agreement": (
                x["label"] == x["human_label"]
            ).mean(),

            "cohen_kappa": cohen_kappa_score(
                x["human_label"],
                x["label"]
            ),
        }
    ])

    display(result)

    return result


print("Human validation sample saved:")
print(human_path)
print("Judge Model:", LLAMA_JUDGE_MODEL)

## **24.Rule Abstraction synthetic query 100건 정확도**

In [ ]:
# Rule Abstraction에서 Qwen을 사용할 경우의 OpenRouter 설정
# 공통 openrouter_client / 하드코딩 OPENROUTER_API_KEY를 그대로 사용합니다.
QWEN_RULE_MODEL = QWEN_MODEL
print("Qwen Rule Abstraction candidate:", QWEN_RULE_MODEL)


In [ ]:
# ============================================================
# HCLA Rule Abstraction Synthetic Query Accuracy Experiment
# Llama 3.3 70B Instruct via OpenRouter
#
# JSON COMPLETELY REMOVED
#
# Output format:
# LOGIC|FIELD|OPERATOR|VALUE;FIELD|OPERATOR|VALUE
#
# Example:
# OR|USD Value|>=|10000;Month|==|7
#
# Evaluation:
# Llama Rule Abstraction
#      ↓
# Canonical Rule String
#      ↓
# Predefined Gold Rule
#      ↓
# Deterministic Python Comparison
# ============================================================

# 최초 1회 필요 시
# !pip -q install -U openai

import time
import random
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from openai import OpenAI


# ============================================================
# 0. CONFIG
# ============================================================

N_SYNTHETIC = 100
SEED = 42

MAX_RETRIES = 5
CALL_INTERVAL = 0.2

random.seed(SEED)
np.random.seed(SEED)


# ============================================================
# 1. OpenRouter API — shared hardcoded key/client
# ============================================================
RULE_MODEL = LLAMA_MODEL
print("Rule Abstraction Model:", RULE_MODEL)

# ============================================================
# 2. RULE ABSTRACTION PROMPT
# ============================================================

RULE_SYSTEM_PROMPT = """
You are the Rule Abstraction component of HCLA.

Convert a natural-language transaction query
into ONE canonical rule string.

Use ONLY these fields:

USD Value
Year
Month
Day
Hour
DayOfWeek

Allowed operators:

>
>=
<
<=
==
between

Allowed logical operators:

AND
OR

OUTPUT FORMAT:

LOGIC|FIELD|OPERATOR|VALUE

For multiple conditions:

LOGIC|FIELD|OPERATOR|VALUE;FIELD|OPERATOR|VALUE

Examples:

Input:
Find transactions of at least $10,000.

Output:
AND|USD Value|>=|10000


Input:
Find transactions between $5,000 and $10,000.

Output:
AND|USD Value|between|5000,10000


Input:
Find transactions of at least $10,000 or
transactions occurring in July.

Output:
OR|USD Value|>=|10000;Month|==|7


Input:
Find transactions above $10,000 and
occurring at hour 8.

Output:
AND|USD Value|>|10000;Hour|==|8


IMPORTANT:

1. Return exactly ONE line.
2. Do NOT return JSON.
3. Do NOT use markdown.
4. Do NOT explain your answer.
5. Do NOT add any text before or after the rule.
6. Preserve AND and OR exactly.
7. Use exact field names.
8. Use exact operators.
"""


# ============================================================
# 3. SYNTHETIC DATASET
# ============================================================

synthetic_queries = []


def add_case(
    case_id,
    category,
    query,
    logic,
    conditions
):

    synthetic_queries.append({
        "case_id": case_id,
        "category": category,
        "query": query,
        "logic": logic,
        "conditions": conditions
    })


case_id = 1


# ============================================================
# A. Single Amount - 20
# ============================================================

amounts = [
    500, 1000, 1500, 2000, 2500,
    3000, 4000, 5000, 6000, 7500,
    8000, 9000, 10000, 12000, 15000,
    20000, 25000, 30000, 40000, 50000
]

single_templates = [
    "Find transactions of at least ${}.",
    "Show transactions greater than or equal to ${}.",
    "Identify transactions worth ${} or more.",
    "Find transactions with a USD value of at least ${}."
]


for amount in amounts:

    template = single_templates[
        (case_id - 1) % len(single_templates)
    ]

    query = template.format(
        f"{amount:,}"
    )

    add_case(
        case_id,
        "SingleAmount",
        query,
        "AND",
        [
            (
                "USD Value",
                ">=",
                amount
            )
        ]
    )

    case_id += 1


# ============================================================
# B. Amount Range - 20
# ============================================================

ranges = [
    (100, 500),
    (500, 1000),
    (1000, 2000),
    (1500, 3000),
    (2000, 5000),
    (3000, 6000),
    (4000, 8000),
    (5000, 10000),
    (6000, 12000),
    (7500, 15000),
    (8000, 16000),
    (10000, 20000),
    (12000, 24000),
    (15000, 30000),
    (20000, 40000),
    (25000, 50000),
    (30000, 60000),
    (40000, 80000),
    (50000, 100000),
    (75000, 150000)
]

range_templates = [
    "Find transactions between ${} and ${}.",
    "Show transactions with USD values from ${} to ${}.",
    "Identify transactions in the ${}–${} range.",
    "Find transactions whose USD value is between ${} and ${}."
]


for low, high in ranges:

    template = range_templates[
        (case_id - 1) % len(range_templates)
    ]

    query = template.format(
        f"{low:,}",
        f"{high:,}"
    )

    add_case(
        case_id,
        "AmountRange",
        query,
        "AND",
        [
            (
                "USD Value",
                "between",
                (low, high)
            )
        ]
    )

    case_id += 1


# ============================================================
# C. Temporal - 20
# ============================================================

years = [
    2020,
    2021,
    2022,
    2023,
    2024
]

month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}


# Year 10개
for i in range(10):

    year = years[
        i % len(years)
    ]

    if i % 2 == 0:

        query = (
            f"Find transactions from {year}."
        )

    else:

        query = (
            f"Show transactions that occurred "
            f"in {year}."
        )

    add_case(
        case_id,
        "Temporal",
        query,
        "AND",
        [
            (
                "Year",
                "==",
                year
            )
        ]
    )

    case_id += 1


# Month 10개
for month in range(1, 11):

    query = (
        f"Find transactions occurring in "
        f"{month_names[month]}."
    )

    add_case(
        case_id,
        "Temporal",
        query,
        "AND",
        [
            (
                "Month",
                "==",
                month
            )
        ]
    )

    case_id += 1


# ============================================================
# D. Compound AND - 20
# ============================================================

for i in range(20):

    amount = 1000 + i * 1000

    year = years[
        i % len(years)
    ]

    query = (
        f"Find transactions of at least "
        f"${amount:,} that occurred "
        f"in {year}."
    )

    add_case(
        case_id,
        "CompoundAND",
        query,
        "AND",
        [
            (
                "USD Value",
                ">=",
                amount
            ),
            (
                "Year",
                "==",
                year
            )
        ]
    )

    case_id += 1


# ============================================================
# E. Compound OR - 10
# ============================================================

for i in range(10):

    amount = 5000 + i * 1000

    month = (
        i % 10
    ) + 1

    query = (
        f"Find transactions of at least "
        f"${amount:,} or transactions "
        f"occurring in "
        f"{month_names[month]}."
    )

    add_case(
        case_id,
        "CompoundOR",
        query,
        "OR",
        [
            (
                "USD Value",
                ">=",
                amount
            ),
            (
                "Month",
                "==",
                month
            )
        ]
    )

    case_id += 1


# ============================================================
# F. Compound Temporal - 10
# ============================================================

for i in range(10):

    amount = 10000 + i * 1000
    hour = 8 + i

    query = (
        f"Find transactions above "
        f"${amount:,} and occurring "
        f"at hour {hour}."
    )

    add_case(
        case_id,
        "CompoundTemporal",
        query,
        "AND",
        [
            (
                "USD Value",
                ">",
                amount
            ),
            (
                "Hour",
                "==",
                hour
            )
        ]
    )

    case_id += 1


assert len(synthetic_queries) == 100

print(
    "\nSynthetic queries:",
    len(synthetic_queries)
)


display(
    pd.DataFrame(
        synthetic_queries
    )[
        [
            "case_id",
            "category",
            "query"
        ]
    ].head(10)
)


# ============================================================
# 4. GOLD RULE → CANONICAL STRING
# ============================================================

def value_to_string(value):

    if isinstance(value, tuple):

        return (
            f"{value[0]},"
            f"{value[1]}"
        )

    return str(value)


def build_canonical_rule(
    logic,
    conditions
):

    condition_strings = []

    for field, operator, value in conditions:

        condition_strings.append(
            f"{field}|"
            f"{operator}|"
            f"{value_to_string(value)}"
        )

    return (
        f"{logic}|"
        +
        ";".join(
            condition_strings
        )
    )


for case in synthetic_queries:

    case["gold_rule"] = (
        build_canonical_rule(
            case["logic"],
            case["conditions"]
        )
    )


# ============================================================
# 5. NORMALIZE MODEL OUTPUT
# ============================================================

def normalize_rule_string(text):

    if text is None:
        return None

    text = str(text).strip()

    # 혹시 markdown을 붙여도 제거
    text = re.sub(
        r"```.*?\n",
        "",
        text
    )

    text = text.replace(
        "```",
        ""
    )

    # 개행 / 탭 제거
    text = text.replace(
        "\n",
        ""
    )

    text = text.replace(
        "\r",
        ""
    )

    text = text.replace(
        "\t",
        ""
    )

    # 불필요한 공백 정리
    text = re.sub(
        r"\s*\|\s*",
        "|",
        text
    )

    text = re.sub(
        r"\s*;\s*",
        ";",
        text
    )

    text = re.sub(
        r"\s*,\s*",
        ",",
        text
    )

    return text.strip()


# ============================================================
# 6. LLAMA RULE ABSTRACTION
# ============================================================

def rule_abstraction(
    query,
    max_retries=MAX_RETRIES
):

    last_error = None

    for attempt in range(
        1,
        max_retries + 1
    ):

        try:

            response = (
                llama_client
                .chat
                .completions
                .create(
                    model=RULE_MODEL,

                    messages=[
                        {
                            "role": "system",
                            "content":
                                RULE_SYSTEM_PROMPT
                        },
                        {
                            "role": "user",
                            "content": query
                        }
                    ],

                    temperature=0
                )
            )

            raw = (
                response
                .choices[0]
                .message
                .content
            )

            normalized = (
                normalize_rule_string(
                    raw
                )
            )

            return {
                "rule": normalized,
                "raw_output": raw,
                "error": None,
                "attempts": attempt
            }

        except Exception as e:

            last_error = str(e)

            error_lower = (
                last_error.lower()
            )

            # 인증/결제 문제는 즉시 중단
            if (
                "401" in last_error
                or "402" in last_error
                or "403" in last_error
                or "credit" in error_lower
                or "insufficient" in error_lower
            ):

                raise

            wait = min(
                2 ** (attempt - 1),
                20
            )

            print(
                f"[Retry] {attempt}/{max_retries} | "
                f"{query[:55]} | "
                f"{last_error[:120]}"
            )

            if attempt < max_retries:

                time.sleep(
                    wait
                )

    return {
        "rule": None,
        "raw_output": None,
        "error": last_error,
        "attempts": max_retries
    }


# ============================================================
# 7. RULE COMPONENT PARSER
#
# JSON parsing 아님.
# "|"와 ";"로 단순 분리해서
# Field / Operator / Value / Logic 정확도 계산
# ============================================================

def parse_rule_components(
    rule
):

    if rule is None:
        return None

    try:

        first_pipe = rule.find("|")

        if first_pipe == -1:
            return None

        logic = (
            rule[:first_pipe]
            .strip()
            .upper()
        )

        remainder = (
            rule[
                first_pipe + 1:
            ]
        )

        condition_texts = (
            remainder.split(";")
        )

        conditions = []

        for c in condition_texts:

            parts = c.split("|")

            if len(parts) != 3:
                return None

            field = (
                parts[0]
                .strip()
            )

            operator = (
                parts[1]
                .strip()
            )

            value = (
                parts[2]
                .strip()
                .replace(",", ",")
            )

            conditions.append(
                (
                    field,
                    operator,
                    value
                )
            )

        return {
            "logic": logic,
            "conditions":
                conditions
        }

    except:

        return None


# ============================================================
# 8. COMPONENT EVALUATION
# ============================================================

def evaluate_rule(
    gold_rule,
    pred_rule
):

    gold_rule = (
        normalize_rule_string(
            gold_rule
        )
    )

    pred_rule = (
        normalize_rule_string(
            pred_rule
        )
    )

    exact_match = int(
        gold_rule
        ==
        pred_rule
    )

    gold = (
        parse_rule_components(
            gold_rule
        )
    )

    pred = (
        parse_rule_components(
            pred_rule
        )
    )

    if pred is None:

        return {
            "format_valid": 0,
            "logic_correct": 0,
            "field_correct": 0,
            "operator_correct": 0,
            "value_correct": 0,
            "exact_match": 0
        }

    format_valid = 1

    logic_correct = int(
        gold["logic"]
        ==
        pred["logic"]
    )

    if (
        len(
            gold["conditions"]
        )
        !=
        len(
            pred["conditions"]
        )
    ):

        return {
            "format_valid":
                format_valid,

            "logic_correct":
                logic_correct,

            "field_correct": 0,
            "operator_correct": 0,
            "value_correct": 0,
            "exact_match":
                exact_match
        }

    field_correct = int(
        all(
            g[0] == p[0]

            for g, p in zip(
                gold["conditions"],
                pred["conditions"]
            )
        )
    )

    operator_correct = int(
        all(
            g[1] == p[1]

            for g, p in zip(
                gold["conditions"],
                pred["conditions"]
            )
        )
    )

    value_correct = int(
        all(
            g[2] == p[2]

            for g, p in zip(
                gold["conditions"],
                pred["conditions"]
            )
        )
    )

    return {
        "format_valid":
            format_valid,

        "logic_correct":
            logic_correct,

        "field_correct":
            field_correct,

        "operator_correct":
            operator_correct,

        "value_correct":
            value_correct,

        "exact_match":
            exact_match
    }


# ============================================================
# 9. RUN
# ============================================================


RULE_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "Rule_Abstraction_100")
os.makedirs(RULE_OUTPUT_DIR, exist_ok=True)

RULE_CKPT = os.path.join(CHECKPOINT_DIR, "rule_abstraction_100_rows.csv")

results_df = load_csv_or_empty(RULE_CKPT)
results = results_df.to_dict("records") if len(results_df) else []
done = set(results_df["case_id"].astype(str)) if len(results_df) else set()

start_time = time.time()

for case in synthetic_queries:
    key = str(case["case_id"])

    if key in done:
        continue

    output = rule_abstraction(case["query"])
    scores = evaluate_rule(case["gold_rule"], output["rule"])

    results.append({
        "case_id": case["case_id"],
        "category": case["category"],
        "query": case["query"],
        "gold_rule": case["gold_rule"],
        "pred_rule": output["rule"],
        "raw_output": output["raw_output"],
        "error": output["error"],
        "attempts": output["attempts"],
        **scores
    })

    done.add(key)
    results_df = pd.DataFrame(results)
    atomic_csv_save(results_df, RULE_CKPT)

    print(f"✅ Rule Abstraction {len(done)}/100")
    time.sleep(CALL_INTERVAL)

results_df = pd.DataFrame(results)
print(f"Rule Abstraction resume complete: {len(results_df)}/100")


# ============================================================
# 10. OVERALL ACCURACY
# ============================================================

metrics = [
    "format_valid",
    "field_correct",
    "operator_correct",
    "value_correct",
    "logic_correct",
    "exact_match"
]

summary = (
    results_df[
        metrics
    ]
    .mean()
    .mul(100)
    .round(2)
)


print(
    "\n========================================"
)

print(
    " Rule Abstraction Accuracy (N=100)"
)

print(
    " Llama 3.3 70B Instruct"
)

print(
    " JSON-free canonical rule format"
)

print(
    "========================================"
)


for metric, value in summary.items():

    print(
        f"{metric:22s}: "
        f"{value:6.2f}%"
    )


# ============================================================
# 11. CATEGORY ACCURACY
# ============================================================

category_summary = (
    results_df
    .groupby(
        "category"
    )[metrics]
    .mean()
    .mul(100)
    .round(2)
)


print(
    "\n=== Accuracy by Query Type ==="
)

display(
    category_summary
)


# ============================================================
# 12. FAILURES
# ============================================================

failures = (
    results_df[
        results_df[
            "exact_match"
        ] == 0
    ]
)


print(
    "\n========================================"
)

print(
    " Failure Summary"
)

print(
    "========================================"
)

print(
    "Exact-match failures:",
    len(failures)
)


if len(failures) > 0:

    pd.set_option(
        "display.max_colwidth",
        None
    )

    display(
        failures[
            [
                "case_id",
                "category",
                "query",
                "gold_rule",
                "pred_rule",
                "raw_output",
                "format_valid",
                "field_correct",
                "operator_correct",
                "value_correct",
                "logic_correct",
                "error"
            ]
        ]
    )

else:

    print(
        "No failures."
    )


# ============================================================
# 13. VISUALIZATION
# ============================================================

plot_metrics = (
    summary[
        [
            "field_correct",
            "operator_correct",
            "value_correct",
            "logic_correct",
            "exact_match"
        ]
    ]
)


plot_metrics.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.ylabel(
    "Accuracy (%)"
)

plt.xlabel(
    "Metric"
)

plt.title(
    "Rule Abstraction Accuracy "
    "on 100 Synthetic Queries\n"
    "Llama 3.3 70B Instruct"
)

plt.ylim(
    0,
    105
)

plt.xticks(
    rotation=30
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# 14. FINAL RESULT
# ============================================================

runtime = (
    time.time()
    -
    start_time
)

print(
    "\n========================================"
)

print(
    " FINAL RESULT"
)

print(
    "========================================"
)

print(
    f"Field Accuracy      : "
    f"{summary['field_correct']:.2f}%"
)

print(
    f"Operator Accuracy   : "
    f"{summary['operator_correct']:.2f}%"
)

print(
    f"Value Accuracy      : "
    f"{summary['value_correct']:.2f}%"
)

print(
    f"Logic Accuracy      : "
    f"{summary['logic_correct']:.2f}%"
)

print(
    f"Exact-Match Accuracy: "
    f"{summary['exact_match']:.2f}%"
)

print(
    f"Total Runtime       : "
    f"{runtime:.1f} sec"
)

# ============================================================
# 15. SAVE PAPER-READY TABLES
# ============================================================
RULE_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "Rule_Abstraction_100")
os.makedirs(RULE_OUTPUT_DIR, exist_ok=True)

results_df.to_csv(
    os.path.join(RULE_OUTPUT_DIR, "rule_abstraction_100_all_cases.csv"),
    index=False, encoding="utf-8-sig"
)
category_summary.reset_index().to_csv(
    os.path.join(RULE_OUTPUT_DIR, "rule_abstraction_100_by_category.csv"),
    index=False, encoding="utf-8-sig"
)
failures.to_csv(
    os.path.join(RULE_OUTPUT_DIR, "rule_abstraction_100_failures.csv"),
    index=False, encoding="utf-8-sig"
)
pd.DataFrame({"metric": summary.index, "accuracy_pct": summary.values}).to_csv(
    os.path.join(RULE_OUTPUT_DIR, "rule_abstraction_100_overall_accuracy.csv"),
    index=False, encoding="utf-8-sig"
)

print("Saved Rule Abstraction paper tables to:", RULE_OUTPUT_DIR)



## 체크포인트 / Resume 적용 완료

`OUTPUT_DIR/_checkpoints`에 완료 건을 계속 저장합니다.

- RQ1: model × experiment × seed
- RQ2: case × condition × rep
- RQ3: case × format × rep
- RQ4: case × condition × rep
- RQ5: case × condition × rep
- Rule Abstraction: case_id

런타임이 끊긴 뒤 위에서부터 다시 실행하면 완료된 건은 자동으로 건너뛰고 남은 건부터 이어서 실행합니다.
완전히 처음부터 다시 돌릴 때만 `_checkpoints`의 해당 파일을 삭제하세요.


## Reviewer-response experiment settings (revised)

- **RQ2 Unsupported Claim / Groundedness:** Full mode uses 100 cases (50 Normal + 50 Anomaly), 2 conditions, and 3 generations = 600 explanations before claim-level decomposition.
- **Rule Abstraction:** 100 synthetic queries with overall, component, category, and failure tables saved as CSV.
- **RQ3:** Explicit mapping: Numerical = Score only; FeatureAttribution = Score + SHAP; Template = Score + SHAP + fixed Template; HCLA = Score + SHAP + evidence-constrained HCLA. All receive the same case/evidence package. Added specificity, evidence utilization, ESR/UCR/CR, and deterministic evidence coverage to avoid relying on tied 1–5 scores.
- **RQ4 HCLA Ablation:** Full mode uses 100 cases × 4 ablation conditions × 3 repetitions = 1,200 rows.
- **RQ5 repeated explanation/stability:** Full mode uses 100 cases × 3 request conditions × 10 repetitions = 3,000 rows, so stability/generalization is no longer based on 3 cases or 3 repetitions.



## 실험 해석 권장 구조

**RQ1a. Detection Performance**  
동일 feature representation / split에서 전통 ML, boosting, tabular DL을 비교합니다.

**RQ1b. Generalization Robustness**  
E1 → E2 → E3 → E4로 조건을 강화하며 성능 감소를 비교합니다. 특히 `macro_f1_drop`, `anomaly_f1_drop`, `pr_auc_drop`을 같이 제시하면 좋습니다.

**SHAP**  
선택된 Evidence Scoring detector의 global/local feature sensitivity를 시각화하고, Top-k SHAP signal을 RQ2 explicit evidence로 사용합니다.

**RQ2**  
Qwen이 생성한 narrative를 Llama가 claim 단위로 `SUPPORTED / UNSUPPORTED / CONTRADICTED` 판정합니다.

**RQ3**  
Numerical / Feature Attribution / Template / HCLA를 비교합니다.

**RQ4**  
Role separation / Evidence constraint / Interactive refinement의 기여를 ablation합니다.

**RQ5**  
반복 생성 안정성, ambiguous query, adversarial instruction 하에서 evidence constraint 유지 여부를 검증합니다.

자동 Llama judge 결과는 일부 human validation과 Cohen's κ를 함께 제시하는 것이 좋습니다.